In [ ]:
import json
import pathlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch

import src.utils as utils
import src.visuals as visual

from src.models import PINN
from src.dataloader import load_data

from IPython.display import Image as IPImage, display

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

TRAIN_DATA_PATH = pathlib.Path("data/sampled_data/frac_1")
TRAIN_FILE_NAME = "data"

TEST_DATA_PATH = pathlib.Path("data/raw_data")
TEST_FILE_NAME = "extrapolation_test.csv"

RUNS_DIR = pathlib.Path("extrapolation_runs")

INPUT_COL_NAMES = ["time", "re", "x", "y"]
TARGET_COL_NAMES = ["U_x", "U_y", "p"]

RE_INDEX = 2       # koji Re iz ekstrapolacionog skupa (po redu sortiranih vrijednosti)
TIME_STEP = 5.0    # vremenski trenutak

In [ ]:
candidates = []

for run in sorted(RUNS_DIR.iterdir()):
    cfg_path = run / "config.json"
    metr_path = run / "metrics.csv"

    if not cfg_path.exists() or not metr_path.exists():
        continue

    cfg = json.load(open(cfg_path))
    metrics = pd.read_csv(metr_path).iloc[0]   # jedan red: metrike na cijelom test skupu

    candidates.append({
        "run": run,
        "c_physics": cfg["physics_weight"],
        "R2_all": float(metrics["R2_all"]),
        "RelL2_all": float(metrics["RelL2_all"]),
    })

cand_df = (
    pd.DataFrame([{k: v for k, v in c.items() if k != "run"} for c in candidates])
    .sort_values("c_physics")
    .reset_index(drop=True)
)

print(f"{len(candidates)} treniranih modela u {RUNS_DIR}")
display(cand_df)

In [ ]:
# Bez fizike = c_physics == 0. Najbolji sa fizikom = max R2 na ekstrapolaciji (c_physics > 0).
no_phys = next(c for c in candidates if c["c_physics"] == 0.0)
best_phys = max((c for c in candidates if c["c_physics"] > 0.0), key=lambda c: c["R2_all"])

print(f"Bez fizike:     {no_phys['run'].name}")
print(f"  c_physics={no_phys['c_physics']}, R2_all={no_phys['R2_all']:.4f}")
print(f"\nPINN (fizika):  {best_phys['run'].name}")
print(f"  c_physics={best_phys['c_physics']}, R2_all={best_phys['R2_all']:.4f}")

In [ ]:
train_df, _, _ = load_data(TRAIN_DATA_PATH, TRAIN_FILE_NAME)

mean = train_df.mean()
std = train_df.std()


def load_pinn(run_dir):
    model = PINN(len(INPUT_COL_NAMES), len(TARGET_COL_NAMES)).to(device)
    ckpt = torch.load(run_dir / "best_model.pth", map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    return model


model_phys = load_pinn(best_phys["run"])
model_nophys = load_pinn(no_phys["run"])

print("Modeli ucitani.")

In [ ]:
test_df = pd.read_csv(TEST_DATA_PATH / TEST_FILE_NAME)

# Sortiranje je NEOPHODNO za reshape u src/visuals.py (y-major, x-minor).
test_df = test_df.sort_values(["re", "time", "y", "x"]).reset_index(drop=True)

re_values = sorted(test_df["re"].unique())
RE_VALUE = re_values[RE_INDEX]

train_re = sorted(train_df["re"].unique())
print(f"Trening Re opseg:    [{min(train_re):.0f}, {max(train_re):.0f}]")
print(f"Ekstrapolacioni Re:  {[round(r, 1) for r in re_values]}")
print(f"Odabrano:            Re = {RE_VALUE:.0f}  (t = {TIME_STEP})")

state = test_df[(test_df["re"] == RE_VALUE) & (test_df["time"] == TIME_STEP)].copy()
print(f"Tacaka u stanju: {len(state)}")

In [ ]:
visual.plot_velocity_and_pressure(test_df, TIME_STEP, RE_VALUE, "Ground truth:")

In [ ]:
pred_phys = utils.predict_field(model_phys, INPUT_COL_NAMES, TARGET_COL_NAMES, state, mean, std, device)
pred_nophys = utils.predict_field(model_nophys, INPUT_COL_NAMES, TARGET_COL_NAMES, state, mean, std, device)

In [ ]:
visual.plot_velocity_and_pressure(
    pred_phys, TIME_STEP, RE_VALUE, f"PINN (fizika, c={best_phys['c_physics']}):"
)

In [ ]:
visual.plot_velocity_and_pressure(pred_nophys, TIME_STEP, RE_VALUE, "Bez fizike (c=0):")

In [ ]:
print("=== PINN (fizika, c={}) ===".format(best_phys["c_physics"]))
visual.compare_predictions(model_phys, test_df, TIME_STEP, RE_VALUE, mean, std, device)

In [ ]:
print("=== Bez fizike (c=0) ===")
visual.compare_predictions(model_nophys, test_df, TIME_STEP, RE_VALUE, mean, std, device)

In [ ]:
rows = []
for name, model in [
    ("Bez fizike (c=0)", model_nophys),
    (f"PINN (fizika, c={best_phys['c_physics']})", model_phys),
]:
    m = utils.evaluate_model(model, state, INPUT_COL_NAMES, TARGET_COL_NAMES, mean, std, device)["all"]
    rows.append({"model": name, **{k: m[k] for k in ["R2", "MAE", "RMSE", "RelL2"]}})

print(f"Re={RE_VALUE:.0f}, t={TIME_STEP}")
display(pd.DataFrame(rows))

In [ ]:
max_train_re = max(train_re)
rows = []
for re in re_values:
    st = test_df[test_df["re"] == re]
    m_phys = utils.evaluate_model(model_phys, st, INPUT_COL_NAMES, TARGET_COL_NAMES, mean, std, device)["all"]
    m_nop = utils.evaluate_model(model_nophys, st, INPUT_COL_NAMES, TARGET_COL_NAMES, mean, std, device)["all"]
    rows.append({
        "re": re,
        "R2_phys": m_phys["R2"], "R2_nophys": m_nop["R2"],
        "RelL2_phys": m_phys["RelL2"], "RelL2_nophys": m_nop["RelL2"],
    })

deg = pd.DataFrame(rows)
display(deg)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(deg["re"], deg["R2_phys"], marker="o", label=f"PINN (fizika, c={best_phys['c_physics']})")
axes[0].plot(deg["re"], deg["R2_nophys"], marker="s", label="Bez fizike (c=0)")
axes[0].set_ylabel(r"$R^2$")

axes[1].plot(deg["re"], deg["RelL2_phys"], marker="o", label=f"PINN (fizika, c={best_phys['c_physics']})")
axes[1].plot(deg["re"], deg["RelL2_nophys"], marker="s", label="Bez fizike (c=0)")
axes[1].set_ylabel("Relative L2")

for ax in axes:
    ax.axvline(max_train_re, color="red", linestyle="--", alpha=0.6, label=f"granica treninga (Re={max_train_re:.0f})")
    ax.set_xlabel("Reynolds")
    ax.grid(alpha=0.3)
    ax.legend()

fig.suptitle("Ekstrapolacija: kvalitet vs Reynoldsov broj", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
ANIM_OUT = pathlib.Path("results/re_extrapolation_truth_vs_pred.gif")
ANIM_OUT.parent.mkdir(parents=True, exist_ok=True)

visual.animate_truth_vs_pred(
    model_phys, test_df, RE_VALUE, mean, std, device,
    fps=2, output_file=str(ANIM_OUT),
)

print(f"GIF spreman: {ANIM_OUT}")
display(IPImage(str(ANIM_OUT)))

In [ ]:
# BOX van vidnog polja -> overlay se odsijeca i ne vidi se (nema held-out regiona kod ekstrapolacije).
NO_BOX = {"x_min": -1.0, "x_max": -1.0, "y_min": -1.0, "y_max": -1.0}

FLOW_ANIM_OUT = pathlib.Path("results/re_extrapolation_flow_grid.gif")
FLOW_ANIM_OUT.parent.mkdir(parents=True, exist_ok=True)

FLOW_ANIM_OUT = visual.animate_flow_tmp(
    FLOW_ANIM_OUT, NO_BOX, test_df, RE_VALUE,
    model_phys, model_nophys, best_phys["c_physics"], mean, std, device,
)

print(f"GIF spreman: {FLOW_ANIM_OUT}")
display(IPImage(str(FLOW_ANIM_OUT)))